# Acupuncture Prescription and Target Prediction Data for Parkinson’s Disease, 2006–2026 Exploration with `mlcroissant`
This notebook guides you in loading, exploring, and processing the FAIR² dataset on acupuncture prescriptions and target prediction using the `mlcroissant` library.

**Dataset Source:**

The dataset is defined via a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.v2mb-pbv9/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load and inspect metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.v2mb-pbv9/fair2.json"

# Load the dataset and access metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset title:", metadata.name)
print("Dataset description:", metadata.description)
print("Published:", metadata.datePublished)
print("Version:", metadata.version)
print("License:", metadata.license)
print("Keywords:", metadata.keywords)


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Croissant schemas organize data into *record sets*, which contain *fields* (columns). Use the `@id` of each entity for referencing.

In [ ]:
# Fetch all record sets from metadata
record_sets = []
if hasattr(metadata, "recordSet"):
    record_sets = list(metadata.recordSet)

print("Available record sets and fields:")
for record_set in record_sets:
    print(f"Record set: {record_set['@id']}  ({record_set.get('name', 'no name')})")
    fields = record_set.get('field', [])
    for field in fields:
        # Each field is a dict with @id, name, dataType etc.
        print(f"   Field: {field['@id']}, Name: {field.get('name','')}, Data type: {field.get('dataType', '')}")

Below, we preview the first record from each available record set. All entities are referenced by their `@id`.

In [ ]:
# Preview first record in each record set
for rset in record_sets:
    rset_id = rset['@id']
    print(f"\nRecords in record set {rset_id}:")
    try:
        records = list(dataset.records(record_set=rset_id))
        # Preview one record
        print(records[0] if records else "No records available.")
    except Exception as e:
        print(f"  Could not load records: {e}")

## 3. Data Extraction
Load data for selected record sets referenced by `@id` into DataFrames for analysis.

Replace `<record set @id>` and field names with actual `@id`s found above.

In [ ]:
# Collect `@id`s of all record sets
record_set_ids = [rset['@id'] for rset in record_sets]
dataframes = {}
for rset_id in record_set_ids:
    records = list(dataset.records(record_set=rset_id))
    df = pd.DataFrame(records)
    dataframes[rset_id] = df
    print(f"Loaded {len(df)} records for Record Set @id: {rset_id}")
    if not df.empty:
        print("Columns:", df.columns.tolist())
        print("Preview:")
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

Reference fields by their `@id`. *Replace the IDs below with actual IDs as appropriate for your record set!*

In [ ]:
# Example: Choose a record set and numeric field by their @id
# (Replace with your actual @id values)
example_record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes.get(example_record_set_id, pd.DataFrame())

# Identify a numeric field (by @id)
numeric_field_id = None
group_field_id = None

# Try to pick sensible defaults from available columns
if not df.empty:
    for col in df.columns:
        # Heuristically select numeric fields
        if df[col].dtype in ["float64", "int64"] and numeric_field_id is None:
            numeric_field_id = col
        elif group_field_id is None and df[col].dtype == "object":
            group_field_id = col

print(f"Numeric field chosen (@id): {numeric_field_id}")
print(f"Grouping field chosen (@id): {group_field_id}")

# Filtering - remove outliers above threshold
threshold = df[numeric_field_id].mean() + df[numeric_field_id].std() if numeric_field_id else None
if threshold:
    filtered = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered.head())

    # Normalization
    filtered[f"{numeric_field_id}_normalized"] = (
        filtered[numeric_field_id] - filtered[numeric_field_id].mean()
    ) / filtered[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping
    if group_field_id in filtered.columns:
        grouped = filtered.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id}:")
        display(grouped.head())

## 5. Visualization
Visualize distributions and relationships between fields using DataFrame columns referenced by their `@id`.
Replace field `@id`s with those appropriate for your record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution
if numeric_field_id and not df.empty:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Relationship between numeric and grouping field
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, overview, extraction, processing, and visualization for the FAIR² acupuncture prescription and target dataset, referencing each entity by its `@id` according to the Croissant schema.

- Key record sets and fields can be accessed by their `@id`.
- EDA and visualization illustrated filtering, normalization, and grouping.
- For further analysis, refer to domain-specific `@id`s and consult the Croissant schema for deeper metadata and provenance.

Explore additional record sets and fields as needed for your research!